In [0]:
catalog = "capgemini_trainingforthecase"

source_path = (
    f"/Volumes/{catalog}/landing/files/"
)

schema_path = (
    f"/Volumes/{catalog}/landing/metadata/sales/schema"
)

checkpoint_path = (
    f"/Volumes/{catalog}/landing/metadata/sales/checkpoint"
)

In [0]:
# As variaveis source_path, schema_path e checkpoint_path
# ja foram definidas na celula 1 — nao precisamos redefini-las.

# Verifica se a tabela ja existe para evitar re-processar dados
try:
    existing_count = spark.table(f"{catalog}.bronze.sales").count()
    print(f"Tabela {catalog}.bronze.sales ja existe com {existing_count} linhas.")
    print("Pulando o Auto Loader — dados ja foram carregados.")
except:
    print("Tabela nao encontrada. Iniciando Auto Loader...")
    
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.pathGlobFilter", "sales*.csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", "true")
        .load(source_path)
    )

    query = (
        df.writeStream
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(f"{catalog}.bronze.sales")
    )
    
    query.awaitTermination()
    print("Auto Loader concluido com sucesso!")